# 1 Installation

In [1]:
%pip install -q "openai>=1.40" pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, hashlib, platform, sys, time, random, re, uuid, hmac
import datetime as dt
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd

DRY_RUN = False   # ← mettez True pour un premier test SANS appel API (résultats simulés)
DRY_RUN = DRY_RUN or os.environ.get("SOC_DRY_RUN", "0") == "1"

# Modèles × architectures évalués 
MODELS = ["gpt-4o-2024-08-06", "gpt-4o-mini-2024-07-18"]
ARCHITECTURES = ["multi_agent", "single_agent"]   # 4 agents séquentiels vs 1 appel unique
N_REPEATS   = 3        # répétitions indépendantes par alerte
TEMPERATURE = 0.0
TOP_P       = 1.0
SEED        = 42       # 'seed' OpenAI = déterminisme best-effort ; system_fingerprint est journalisé
MAX_TOKENS  = {"triage": 400, "mitre": 500, "investigation": 900, "reporting": 1200, "single": 2500}

ALERTS_PATH = "alerts_v2.csv"   # 50 alertes d'origine annotées + 25 négatifs difficiles
HUMAN_MEASUREMENTS_PATH = "human_measurements.csv"   # optionnel

# RUN_ID FIXE => relancer le notebook reprend là où il s'était arrêté (les traitements faits ne sont pas refaits)
RUN_ID  = os.environ.get("SOC_RUN_ID") or ("dryrun_test" if DRY_RUN else "run_final")
OUT_DIR = Path("outputs") / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
RNG = np.random.default_rng(SEED)
print(f"Dossier de travail : {Path.cwd()}")
print(f"RUN_ID={RUN_ID} | DRY_RUN={DRY_RUN} | modèles={MODELS} | architectures={ARCHITECTURES} | répétitions={N_REPEATS}")
if DRY_RUN:
    print("⚠️  MODE SIMULÉ : les chiffres produits ne sont PAS des mesures.")

Dossier de travail : C:\Users\a847071\Downloads\Inputs
RUN_ID=run_final | DRY_RUN=False | modèles=['gpt-4o-2024-08-06', 'gpt-4o-mini-2024-07-18'] | architectures=['multi_agent', 'single_agent'] | répétitions=3


# 3 Paramètres d'estimation : valeur centrale, plage et source

Chaque paramètre a une valeur centrale (`central`) et une plage (`low` / `high`), utilisées par le Monte Carlo et l'analyse de sensibilité.

In [3]:
PARAMS = {

    # --- Estimateur E1 : énergie par requête (benchmark publié) ---------------------------
    "E_QUERY_WH":        dict(central=0.31, low=0.16, high=0.60, unit="Wh/requête", includes_pue=True,
                              source="Réf. [6] : Oviedo et al., 'Energy use of AI inference, efficiency pathways, and test-time scaling', Joule, 2026, doi:10.1016/j.joule.2026.102430 (arXiv:2509.20241) : médiane 0,31 Wh/requête (IQR 0,16–0,60), requête type 500 tokens entrée / 300 sortie ; PUE INCLUS (log-normale P5–P95 1,05–1,40)",
                              note="Plage = IQR publié. Pour GPT-4o mini, multiplié par le rapport Jegham [12] mini/4o (E1_FACTOR)."),
    # --- Estimateur E2 : énergie par token, source INDÉPENDANTE de [6] ---------------------
    "E_OUT_J_PER_TOKEN": dict(central=3.5, low=1.0, high=4.0, unit="J/token de sortie", includes_pue=False,
                              source="Réf. [10] : Samsi et al., 'From Words to Watts: Benchmarking the Energy Costs of Large Language Model Inference', IEEE HPEC 2023 : ≈ 3–4 J par token de sortie, LLaMA-65B, énergie GPU seule (sans PUE)",
                              note="Borne basse 1 J : matériel plus récent et batching en production (HYPOTHÈSE)."),
    "PREFILL_RATIO":     dict(central=0.13, low=0.05, high=0.30, unit="ratio",
                              source="Dérivé de la réf. [12] (Jegham et al., Table 4 : ajustement linéaire sur les 3 tailles de prompt GPT-4o : ≈ 0,47 J/token entrée vs 3,5 J/token sortie) — calcul reproduit au §3/§12",
                              note="Énergie d'un token d'entrée / énergie d'un token de sortie."),
    "PUE":               dict(central=1.12, low=1.09, high=1.54, unit="-",
                              source="Central : PUE Azure 1,12 (réf. [12], Table 1, d'après Microsoft). Haut : moyenne mondiale 1,54, Uptime Institute Global Data Center Survey 2025 (UII Keynote Report 180, juillet 2025) — remplace la réf. [37] erronée",
                              note="Appliqué UNIQUEMENT aux coefficients avec includes_pue=False (ici E2)."),
    # --- Intensités carbone séparées (R2) --------------------------------------------------
    "CI_CLOUD":          dict(central=350, low=250, high=435, unit="gCO2e/kWh",
                              source="0,35 kgCO2e/kWh pour l'inférence OpenAI sur Azure (réf. [12], Table 1, d'après le rapport de durabilité Microsoft) ; haut = moyenne mondiale IEA 2025 (réf. [41])",
                              note="Réseau du datacenter (inférence)."),
    "CI_WORKSTATION":    dict(central=435, low=50, high=750, unit="gCO2e/kWh",
                              source="Réf. [41] : IEA, Electricity 2026, chapitre Emissions : intensité mondiale moyenne = 435 gCO2/kWh en 2025, 360 en 2030 (445 en 2024 selon IEA Electricity 2025). Maroc : 536 gCO2e/kWh (LowCarbonPower.org, données Ember, avr. 2025–mars 2026)",
                              note="Réseau du pays du SOC (postes analystes). Plage = réseaux bas carbone → fortement carbonés."),
    # --- Baseline humaine (R1.1, R2) -------------------------------------------------------
    "WS_POWER_W":        dict(central=120, low=70, high=250, unit="W",
                              source="Ordinateur de bureau actif 36–250 W (moyenne 55 W) et écran LCD 15–35 W : synthèse DSSW 'Review of Computer Energy Consumption' (Kawamoto et al. 2001/2004, Roberson et al. 2002). Central = 1 PC + 2 écrans. IDÉAL : mesure au wattmètre",
                              note="Poste + écrans en usage actif."),
    "WS_ATTRIBUTION":    dict(central=1.0, low=0.5, high=1.0, unit="fraction",
                              source="HYPOTHÈSE de modélisation (charge partagée), testée en sensibilité",
                              note="Part du poste attribuable à UNE alerte (<1 = investigations en parallèle)."),
    "HUMAN_MIN_TRIAGE_L1":   dict(central=20, low=10, high=45, unit="min/alerte",
                              source="HYPOTHÈSE (scénario conservateur, triage L1 seul), testée en sensibilité ; à remplacer par une mesure si possible",
                              note="Scénario conservateur : triage niveau 1 seulement."),
    "HUMAN_MIN_FULL_INVEST": dict(central=70, low=30, high=240, unit="min/alerte",
                              source="'The State of AI in the SOC 2025' (enquête auprès de 282 responsables sécurité, rapportée par The Hacker News, sept. 2025) : 70 min en moyenne pour investiguer complètement une alerte. Haut = 4 h (borne haute de la source 2–4 h du manuscrit)"),
    "HUMAN_REVIEW_MIN":  dict(central=15, low=5, high=30, unit="min/alerte",
                              source="HYPOTHÈSE (relecture humaine du rapport IA), testée en sensibilité",
                              note="Relecture humaine résiduelle (scénario hybride)."),
    "CLIENT_W":          dict(central=30, low=10, high=60, unit="W",
                              source="HYPOTHÈSE (ordinateur portable pendant la latence API) ; impact négligeable",
                              note="Machine qui orchestre le pipeline."),
    # --- SOAR & extrapolations -------------------------------------------------------------
    "SOAR_CPU_W":        dict(central=30, low=10, high=65, unit="W",
                              source="HYPOTHÈSE (puissance CPU d'un serveur attribuée au playbook) ; le temps CPU est mesuré (11)"),
    "ALERTS_PER_YEAR":   dict(central=10_000, low=10_000, high=10_000, unit="alertes/an",
                              source="Unité de normalisation (résultats exprimés pour 10 000 alertes investiguées) ; ordre de grandeur réel : 960 alertes/jour en moyenne (State of AI in the SOC 2025)",
                              note="Fixe : le lecteur met à l'échelle selon son volume."),
    "REASONING_TOKEN_MULT": dict(central=8, low=3, high=20, unit="x tokens de sortie",
                              source="HYPOTHÈSE (PROJECTION, non mesurée)",
                              note="Extended reasoning."),
}
N_QUERIES_THEORETICAL = 12   # modèle théorique du manuscrit, présenté séparément

# Benchmark par modèle : Jegham et al., arXiv:2505.09598, Table 4 — [tokens entrée, tokens sortie, Wh/requête], PUE 1,12 inclus.
# NB : Jegham estime GPT-4o mini PLUS énergivore que GPT-4o, car il le suppose servi sur DGX A100 (vs H100 pour GPT-4o).
#      C'est une hypothèse matérielle de la source, à mentionner si vous comparez les deux modèles.
JEGHAM_T4 = {
    "gpt-4o-2024-08-06":      [[100, 300, 0.423], [1000, 1000, 1.215], [10000, 1500, 2.875]],
    "gpt-4o-mini-2024-07-18": [[100, 300, 0.577], [1000, 1000, 1.897], [10000, 1500, 3.098]],
}
REF_MODEL = "gpt-4o-2024-08-06"   # E1_FACTOR = benchmark Jegham [12] du modèle / celui de GPT-4o (prompt court)
E1_FACTOR = {m: t[0][2] / JEGHAM_T4[REF_MODEL][0][2] for m, t in JEGHAM_T4.items()}
JFIT = {m: np.linalg.solve(np.c_[np.array(t)[:, :2], np.ones(3)], np.array(t)[:, 2]) for m, t in JEGHAM_T4.items()}
for m in MODELS:
    if m not in JEGHAM_T4:
        print(f"⚠️  {m} absent de JEGHAM_T4 : E1 utilisera les coefficients de {REF_MODEL} (à compléter).")
    else:
        a, b, c = JFIT[m]
        print(f"{m}: E1 = {JEGHAM_T4[m][0][2]} Wh/requête courte | ajustement : {a*3600:.2f} J/token entrée, "
              f"{b*3600:.2f} J/token sortie, {c:.3f} Wh fixe")
print("E2 (Samsi et al.) est indépendant du modèle : il compare les ARCHITECTURES, pas GPT-4o vs mini.")

CENTRAL = {k: v["central"] for k, v in PARAMS.items()}
params_df = pd.DataFrame(PARAMS).T[["central", "low", "high", "unit", "source", "note"]]
missing = params_df.index[params_df["source"].str.strip() == ""].tolist()
print(f"⚠️  {len(missing)} paramètre(s) sans source : {missing}" if missing else "✅ Toutes les sources sont renseignées.")
params_df

gpt-4o-2024-08-06: E1 = 0.423 Wh/requête courte | ajustement : 0.47 J/token entrée, 3.47 J/token sortie, 0.121 Wh fixe
gpt-4o-mini-2024-07-18: E1 = 0.577 Wh/requête courte | ajustement : 0.11 J/token entrée, 6.65 J/token sortie, 0.020 Wh fixe
E2 (Samsi et al.) est indépendant du modèle : il compare les ARCHITECTURES, pas GPT-4o vs mini.
✅ Toutes les sources sont renseignées.


,central,low,high,unit,source,note
E_QUERY_WH,0.31,0.16,0.6,Wh/requête,"Réf. [6] : Oviedo et al., 'Energy use of AI in...","Plage = IQR publié. Pour GPT-4o mini, multipli..."
E_OUT_J_PER_TOKEN,3.5,1.0,4.0,J/token de sortie,"Réf. [10] : Samsi et al., 'From Words to Watts...",Borne basse 1 J : matériel plus récent et batc...
PREFILL_RATIO,0.13,0.05,0.3,ratio,"Dérivé de la réf. [12] (Jegham et al., Table 4...",Énergie d'un token d'entrée / énergie d'un tok...
PUE,1.12,1.09,1.54,-,"Central : PUE Azure 1,12 (réf. [12], Table 1, ...",Appliqué UNIQUEMENT aux coefficients avec incl...
CI_CLOUD,350,250,435,gCO2e/kWh,"0,35 kgCO2e/kWh pour l'inférence OpenAI sur Az...",Réseau du datacenter (inférence).
CI_WORKSTATION,435,50,750,gCO2e/kWh,"Réf. [41] : IEA, Electricity 2026, chapitre Em...",Réseau du pays du SOC (postes analystes). Plag...
WS_POWER_W,120,70,250,W,Ordinateur de bureau actif 36–250 W (moyenne 5...,Poste + écrans en usage actif.
WS_ATTRIBUTION,1.0,0.5,1.0,fraction,"HYPOTHÈSE de modélisation (charge partagée), t...",Part du poste attribuable à UNE alerte (<1 = i...
HUMAN_MIN_TRIAGE_L1,20,10,45,min/alerte,"HYPOTHÈSE (scénario conservateur, triage L1 se...",Scénario conservateur : triage niveau 1 seulem...
HUMAN_MIN_FULL_INVEST,70,30,240,min/alerte,'The State of AI in the SOC 2025' (enquête aup...,Remplace 250 min (non justifié).


In [ ]:
class _MockCompletions:
    """Faux client pour tester le notebook sans appel API (DRY_RUN). Résultats NON publiables."""
    KEYWORDS = {"mimikatz": "T1003", "powershell": "T1059", "encoded": "T1027", "lsass": "T1003",
                "brute": "T1110", "failed login": "T1110", "phishing": "T1566", "macro": "T1204",
                "rdp": "T1021", "exfil": "T1041", "scheduled task": "T1053", "ransom": "T1486"}

    def __init__(self, seed):
        self.rng = np.random.default_rng(seed)

    def create(self, **kw):
        sys_p, user = kw["messages"][0]["content"], kw["messages"][1]["content"]
        agent = next(a for a, p in PROMPTS.items() if p == sys_p)
        alert = user.split("### ALERT\n", 1)[1].split("\n\n### ", 1)[0].lower()
        hits = [t for k, t in self.KEYWORDS.items() if k in alert]
        mal = bool(hits) ^ (self.rng.random() < 0.1)
        if agent == "triage":
            content = json.dumps({"verdict": "malicious" if mal else "benign",
                                  "severity": "high" if mal else "low",
                                  "confidence": round(float(self.rng.uniform(.6, .95)), 2),
                                  "rationale": "simulated"})
        elif agent == "mitre":
            content = json.dumps({"techniques": [{"id": t, "name": "sim", "tactic": "sim", "justification": "sim"}
                                                 for t in sorted(set(hits))]})
        elif agent == "single":
            content = json.dumps({"verdict": "malicious" if mal else "benign", "severity": "high" if mal else "low",
                                  "confidence": round(float(self.rng.uniform(.6, .95)), 2), "rationale": "simulated",
                                  "techniques": [{"id": t, "name": "sim", "tactic": "sim", "justification": "sim"}
                                                 for t in sorted(set(hits))],
                                  "investigation": " ".join(["lorem"] * int(self.rng.integers(150, 300))),
                                  "report": " ".join(["lorem"] * int(self.rng.integers(150, 350)))})
        else:
            content = " ".join(["lorem"] * int(self.rng.integers(200, 600)))
        pt = int(len((sys_p + " " + user).split()) * 1.3)
        ct = max(20, int(len(content.split()) * 1.3))
        return SimpleNamespace(
            model=kw["model"], system_fingerprint="fp_mock",
            choices=[SimpleNamespace(message=SimpleNamespace(content=content), finish_reason="stop")],
            usage=SimpleNamespace(prompt_tokens=pt, completion_tokens=ct, total_tokens=pt + ct,
                                  prompt_tokens_details=None))


if DRY_RUN:
    client = SimpleNamespace(chat=SimpleNamespace(completions=_MockCompletions(SEED)))
    RETRYABLE = (TimeoutError,)
else:
    from openai import OpenAI, RateLimitError, APITimeoutError, APIConnectionError, InternalServerError
    if not os.environ.get("OPENAI_API_KEY"):
        from getpass import getpass   # la clé est saisie dans une case masquée, jamais écrite dans le notebook
        os.environ["OPENAI_API_KEY"] = getpass("Collez votre clé API OpenAI puis Entrée : ").strip()
    if not os.environ["OPENAI_API_KEY"].startswith("sk-"):
        bad = os.environ.pop("OPENAI_API_KEY")
        raise ValueError(f"Ce texte n'est pas une clé OpenAI (il commence par « {bad[:8]}… », une clé commence par « sk- »). "
                         "Copiez la clé depuis platform.openai.com, puis relancez cette cellule.")
    client = OpenAI()
    RETRYABLE = (RateLimitError, APITimeoutError, APIConnectionError, InternalServerError)
    from openai import AuthenticationError
    try:   # vérification gratuite de la clé avant de lancer quoi que ce soit
        client.models.retrieve(MODELS[0])
        print("Clé API valide, modèle accessible.")
    except AuthenticationError:
        os.environ.pop("OPENAI_API_KEY", None)
        raise RuntimeError("Clé API refusée par OpenAI (401). Relancez cette cellule et collez la bonne clé.") from None
    except Exception as e:   # clé restreinte sans droit de lecture des modèles, réseau… : on n'empêche pas le run
        print(f"(Vérification du modèle impossible : {type(e).__name__} — on continue ; une vraie erreur de clé arrêtera le §8.)")
print("Client prêt.")

In [ ]:
PROMPT_VERSION = "v2.0"
PROMPTS = {
    "triage": (
        "You are a Tier-1 SOC analyst. Classify the security alert.\n"
        "Return ONLY a JSON object with keys:\n"
        '  "verdict": "malicious" or "benign",\n'
        '  "severity": "low" | "medium" | "high" | "critical",\n'
        '  "confidence": number between 0 and 1,\n'
        '  "rationale": string (max 60 words).'),
    "mitre": (
        "You are a threat-intelligence analyst. Map the alert and the triage findings to MITRE ATT&CK Enterprise.\n"
        'Return ONLY a JSON object: {"techniques": [{"id": "T####[.###]", "name": "...", "tactic": "...", '
        '"justification": "..."}]}. At most 5 techniques; return an empty list if none applies.'),
    "investigation": (
        "You are a SOC investigator. Using the alert, the triage and the MITRE mapping, describe the "
        "investigation steps, the evidence to collect, the likely scope and the containment actions. "
        "Max 350 words."),
    "reporting": (
        "You are a SOC lead. Write the final incident report: summary, verdict, ATT&CK techniques, impact, "
        "recommended actions. Max 400 words."),
    # Architecture alternative : un seul agent fait les 4 étapes en un appel (R1.5)
    "single": (
        "You are a senior SOC analyst. For the security alert, perform triage, MITRE ATT&CK Enterprise mapping, "
        "investigation and reporting in ONE answer. Return ONLY a JSON object with keys:\n"
        '  "verdict": "malicious" or "benign",\n'
        '  "severity": "low" | "medium" | "high" | "critical",\n'
        '  "confidence": number between 0 and 1,\n'
        '  "rationale": string (max 60 words),\n'
        '  "techniques": [{"id": "T####[.###]", "name": "...", "tactic": "...", "justification": "..."}] (max 5, may be empty),\n'
        '  "investigation": string (max 350 words: steps, evidence, scope, containment),\n'
        '  "report": string (max 400 words: summary, verdict, techniques, impact, recommended actions).'),
}
JSON_AGENTS = {"triage", "mitre", "single"}
AGENTS = ["triage", "mitre", "investigation", "reporting"]
ALL_AGENTS = AGENTS + ["single"]

(OUT_DIR / "prompts.json").write_text(json.dumps(
    {"prompt_version": PROMPT_VERSION, "prompts": PROMPTS, "max_tokens": MAX_TOKENS,
     "temperature": TEMPERATURE, "top_p": TOP_P, "seed": SEED, "models": MODELS, "architectures": ARCHITECTURES}, indent=2, ensure_ascii=False), encoding="utf-8")
print("Prompts exportés →", OUT_DIR / "prompts.json")

In [ ]:
CALL_LOG    = OUT_DIR / "llm_calls.jsonl"
OUTPUT_LOG  = OUT_DIR / "alert_outputs.jsonl"
FAILURE_LOG = OUT_DIR / "failures.jsonl"


def _append(path, rec):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def llm_call(agent, model, user_content, meta, max_retries=6):
    messages = [{"role": "system", "content": PROMPTS[agent]}, {"role": "user", "content": user_content}]
    kwargs = dict(model=model, messages=messages, temperature=TEMPERATURE, top_p=TOP_P,
                  seed=SEED, max_tokens=MAX_TOKENS[agent])
    if agent in JSON_AGENTS:
        kwargs["response_format"] = {"type": "json_object"}
    for attempt in range(1, max_retries + 1):
        t0 = time.perf_counter()
        try:
            r = client.chat.completions.create(**kwargs)
            break
        except RETRYABLE as e:
            wait = min(60, 2 ** attempt) + random.random()   # backoff exponentiel
            print(f"  [{agent}] {type(e).__name__} (tentative {attempt}/{max_retries}) → pause {wait:.0f}s")
            time.sleep(0 if DRY_RUN else wait)
    else:
        raise RuntimeError(f"Échec après {max_retries} tentatives ({agent})")
    latency = time.perf_counter() - t0
    u = r.usage
    details = getattr(u, "prompt_tokens_details", None)
    cached = getattr(details, "cached_tokens", 0) if details is not None else 0
    content = r.choices[0].message.content or ""
    rec = {**meta, "agent": agent, "model_requested": model, "model_returned": r.model,
           "system_fingerprint": getattr(r, "system_fingerprint", None),
           "prompt_tokens": u.prompt_tokens, "completion_tokens": u.completion_tokens,
           "total_tokens": u.total_tokens, "cached_prompt_tokens": cached or 0,
           "latency_s": latency, "attempts": attempt, "finish_reason": r.choices[0].finish_reason,
           "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
           "messages": messages, "response": content}
    _append(CALL_LOG, rec)
    return content


def build_context(alert, **sections):
    parts = [f"### ALERT\n{alert}"]
    parts += [f"### {k.upper()}\n{v}" for k, v in sections.items()]
    return "\n\n".join(parts)


def process_alert(alert, model, meta):
    """Pipeline Triage → MITRE → Investigation → Reporting (4 requêtes LLM, séquentielles)."""
    tri = llm_call("triage", model, build_context(alert), meta)
    mit = llm_call("mitre", model, build_context(alert, triage=tri), meta)
    inv = llm_call("investigation", model, build_context(alert, triage=tri, mitre_mapping=mit), meta)
    rep = llm_call("reporting", model, build_context(alert, triage=tri, mitre_mapping=mit, investigation=inv), meta)
    return {"triage": tri, "mitre": mit, "investigation": inv, "report": rep}


def process_alert_single(alert, model, meta):
    """Architecture alternative : 1 seule requête LLM qui produit les 4 sorties."""
    raw = llm_call("single", model, build_context(alert), meta)
    try:
        d = json.loads(raw)
    except Exception:
        m = re.search(r"\{.*\}", raw or "", re.S)
        d = json.loads(m.group(0)) if m else {}
    tri = json.dumps({k: d.get(k) for k in ("verdict", "severity", "confidence", "rationale")})
    mit = json.dumps({"techniques": d.get("techniques", [])})
    return {"triage": tri, "mitre": mit, "investigation": str(d.get("investigation", "")),
            "report": str(d.get("report", ""))}


PIPELINES = {"multi_agent": process_alert, "single_agent": process_alert_single}

In [ ]:
def _synthetic_alerts():
    rows = [
        ("Process mimikatz.exe accessed lsass memory on host WS-0142.corp user=jdoe", "malicious", "T1003", "credential"),
        ("PowerShell launched with encoded command from winword.exe on 10.1.4.22", "malicious", "T1059;T1027;T1204", "execution"),
        ("50 failed login attempts for admin from 185.22.14.9 within 2 minutes", "malicious", "T1110", "brute_force"),
        ("User reported phishing email with macro attachment from billing@supplier-x.com", "malicious", "T1566;T1204", "phishing"),
        ("Scheduled task created by IT deployment tool SCCM on 300 hosts", "benign", "", "admin"),
        ("Antivirus signature update completed on SRV-DB-01.corp", "benign", "", "noise"),
        ("RDP connection from jump host 10.0.0.5 to SRV-APP-03.corp by user=svc_backup", "benign", "", "admin"),
        ("Outbound transfer of 4 GB to unknown cloud storage, possible exfil, from 10.2.3.44", "malicious", "T1041", "exfiltration"),
        ("Multiple files renamed with .lock extension, ransom note dropped on FS-02.corp", "malicious", "T1486", "ransomware"),
        ("Vulnerability scanner Nessus scan from 10.9.9.9 (authorized window)", "benign", "", "noise"),
        ("Password reset by helpdesk for user=asmith after ticket INC-4411", "benign", "", "admin"),
        ("Failed login burst for 20 accounts from 45.83.1.7 (password spraying)", "malicious", "T1110", "brute_force"),
    ]
    return pd.DataFrame(rows, columns=["alert", "label", "mitre_gt", "category"]).assign(source="synthetic")


if Path(ALERTS_PATH).exists():
    alerts_df = pd.read_csv(ALERTS_PATH, encoding="utf-8")
    dataset_sha256 = hashlib.sha256(Path(ALERTS_PATH).read_bytes()).hexdigest()
elif DRY_RUN:
    alerts_df = _synthetic_alerts()
    dataset_sha256 = "synthetic"
else:
    raise FileNotFoundError(ALERTS_PATH)

assert "alert" in alerts_df.columns, "Colonne 'alert' manquante"
if "alert_id" not in alerts_df.columns:
    alerts_df.insert(0, "alert_id", [f"A{i:03d}" for i in range(len(alerts_df))])
alerts_df["alert_id"] = alerts_df["alert_id"].astype(str)
assert alerts_df["alert_id"].is_unique, "alert_id doit être unique"
HAS_LABEL = "label" in alerts_df.columns and alerts_df["label"].notna().any()
HAS_MITRE = "mitre_gt" in alerts_df.columns
alerts_df["alert_chars"] = alerts_df["alert"].str.len()

print(f"{len(alerts_df)} alertes | SHA-256 = {dataset_sha256}")
print(f"Vérité terrain verdict : {'oui' if HAS_LABEL else 'NON → qualité non évaluable (R2)'} | MITRE : {'oui' if HAS_MITRE else 'non'}")
for col in ["category", "label", "source"]:
    if col in alerts_df.columns:
        print(f"\nRépartition par {col} :\n{alerts_df[col].value_counts(dropna=False).to_string()}")
print("\nLongueur des alertes (caractères) :\n", alerts_df["alert_chars"].describe().round(1).to_string())

# ---- Pseudonymisation pour publication (à relire manuellement) ----
ANON_SALT = os.environ.get("ANON_SALT", "change-me").encode()
def _pseudo(kind, value):
    return f"<{kind}_{hmac.new(ANON_SALT, value.encode(), 'sha256').hexdigest()[:8]}>"
ANON_PATTERNS = [
    ("EMAIL", re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")),
    ("IP",    re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")),
    ("HOST",  re.compile(r"\b[\w-]+\.(?:corp|local|lan|internal|intra)\b", re.I)),
    ("USER",  re.compile(r"(?<=user=)[\w.$-]+", re.I)),
]
def anonymize(text):
    for kind, pat in ANON_PATTERNS:
        text = pat.sub(lambda m: _pseudo(kind, m.group(0)), text)
    return text

alerts_df.assign(alert=alerts_df["alert"].map(anonymize)).drop(columns="alert_chars") \
         .to_csv(OUT_DIR / "alerts_anonymized.csv", index=False)
print("\nVersion pseudonymisée →", OUT_DIR / "alerts_anonymized.csv", "(À RELIRE avant publication)")

## 8 Exécution : modèles × architectures × répétitions × alertes
L'ordre des alertes est mélangé à chaque répétition, avec une graine fixe. Cela limite les biais liés à l'heure ou à la charge de l'API.

Reprise automatique. Si le notebook s'interrompt, relancez-le simplement depuis le même dossier (RUN_ID fixe = `run_final`) : les alertes déjà traitées sont sautées.

Une alerte en échec n'est pas ignorée silencieusement. Elle est journalisée dans `failures.jsonl`, et les tokens des tentatives partielles sont comptés à part comme surcoût.

In [ ]:
done = set()
if OUTPUT_LOG.exists():
    done = {(r["model"], r["repeat"], r["alert_id"])
            for r in map(json.loads, OUTPUT_LOG.read_text(encoding="utf-8").splitlines())}

# "model" dans les journaux = CONFIGURATION (modèle API + architecture), ex. "gpt-4o-2024-08-06_multi_agent"
n_total = len(MODELS) * len(ARCHITECTURES) * N_REPEATS * len(alerts_df)
print(f"{len(done)}/{n_total} traitements déjà présents (reprise) dans {OUT_DIR.resolve()}")
if not DRY_RUN and len(done) == 0 and os.environ.get("SOC_CONFIRM_NEW_RUN") != "1":
    raise RuntimeError("Aucun traitement existant trouvé : ce lancement referait TOUT le run (coût complet). "
                       "Vérifiez que le notebook est dans le dossier qui contient outputs/run_final. "
                       "Pour un tout nouveau run volontaire, exécutez d'abord : os.environ['SOC_CONFIRM_NEW_RUN'] = '1'")
t_start = time.time()
for api_model in MODELS:
  for arch in ARCHITECTURES:
    model = f"{api_model}_{arch}"
    for rep in range(1, N_REPEATS + 1):
        order = alerts_df.sample(frac=1, random_state=SEED + rep)
        for _, row in order.iterrows():
            key = (model, rep, row["alert_id"])
            if key in done:
                continue
            meta = {"run_id": RUN_ID, "model": model, "api_model": api_model, "arch": arch, "repeat": rep,
                    "alert_id": row["alert_id"], "attempt_id": uuid.uuid4().hex}
            try:
                out = PIPELINES[arch](row["alert"], api_model, meta)
                _append(OUTPUT_LOG, {**meta, **out})
                done.add(key)
            except Exception as e:
                if type(e).__name__ in ("AuthenticationError", "PermissionDeniedError", "NotFoundError", "BadRequestError"):
                    raise   # erreur de clé / d'accès : inutile de continuer sur les autres alertes
                print(f"✗ {key} : {type(e).__name__}: {e}")
                _append(FAILURE_LOG, {**meta, "error": f"{type(e).__name__}: {e}"})
        print(f"{model} — répétition {rep}/{N_REPEATS} terminée ({time.time()-t_start:.0f}s)")
print(f"Terminé : {len(done)}/{n_total} traitements réussis.")

## 9 Tokens mesurés : par alerte, par agent, entrée et sortie, variabilité inter-runs (MESURÉ)

In [ ]:
calls   = pd.read_json(CALL_LOG, lines=True, encoding="utf-8")
outputs = pd.read_json(OUTPUT_LOG, lines=True, encoding="utf-8")
outputs["alert_id"] = outputs["alert_id"].astype(str)
calls["alert_id"]   = calls["alert_id"].astype(str)
KEYS = ["model", "repeat", "alert_id"]

def _split_config(c):
    for a in ARCHITECTURES:
        if c.endswith("_" + a):
            return c[: -len(a) - 1], a
    return c, "multi_agent"
for _df in (calls, outputs):
    _df["api_model"], _df["arch"] = zip(*_df["model"].map(_split_config))
ok_attempts = set(outputs["attempt_id"])
calls_ok    = calls[calls["attempt_id"].isin(ok_attempts)]
calls_waste = calls[~calls["attempt_id"].isin(ok_attempts)]
print(f"Appels LLM : {len(calls)} (dont {len(calls_waste)} issus de tentatives incomplètes = surcoût, "
      f"{calls_waste['total_tokens'].sum():,} tokens)")
print("Modèle(s) renvoyé(s) :", calls["model_returned"].unique().tolist(),
      "| system_fingerprint :", calls["system_fingerprint"].dropna().unique().tolist()[:5])
trunc = (calls_ok["finish_reason"] == "length").mean()
if trunc > 0:
    print(f"⚠️  {trunc:.1%} des réponses tronquées par max_tokens → envisager d'augmenter MAX_TOKENS")

per_alert = calls_ok.groupby(KEYS).agg(
    n_queries=("agent", "size"), prompt_tokens=("prompt_tokens", "sum"),
    completion_tokens=("completion_tokens", "sum"), total_tokens=("total_tokens", "sum"),
    cached_prompt_tokens=("cached_prompt_tokens", "sum"), latency_s=("latency_s", "sum")).reset_index()
pivot = calls_ok.pivot_table(index=KEYS, columns="agent",
                             values=["prompt_tokens", "completion_tokens"], aggfunc="sum")
pivot.columns = [f"{v}_{a}" for v, a in pivot.columns]
per_alert = per_alert.merge(pivot.reset_index(), on=KEYS)
per_alert["api_model"], per_alert["arch"] = zip(*per_alert["model"].map(_split_config))
per_alert["e1_factor"] = per_alert["api_model"].map(E1_FACTOR).fillna(1.0)


def describe(x, n_boot=5000):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    boots = RNG.choice(x, (n_boot, len(x)), replace=True).mean(axis=1)
    return pd.Series({"n": len(x), "mean": x.mean(), "std": x.std(ddof=1) if len(x) > 1 else np.nan,
                      "min": x.min(), "median": np.median(x), "max": x.max(),
                      "ci95_low": np.percentile(boots, 2.5), "ci95_high": np.percentile(boots, 97.5)})


token_cols = ["n_queries", "prompt_tokens", "completion_tokens", "total_tokens", "latency_s"]
token_stats = pd.concat({m: g[token_cols].apply(describe).T for m, g in per_alert.groupby("model")},
                        names=["model", "metric"])
print("\n=== Statistiques par alerte (MESURÉ ; IC95 % bootstrap de la moyenne) ===")
display(token_stats.round(1))

agent_stats = calls_ok.groupby(["model", "agent"])[["prompt_tokens", "completion_tokens"]].mean() \
                      .reindex(ALL_AGENTS, level="agent")
agent_stats["share_of_total_%"] = 100 * agent_stats.sum(axis=1) / agent_stats.sum(axis=1).groupby("model").transform("sum")
print("\n=== Tokens moyens par agent (entrée / sortie séparées) ===")
display(agent_stats.round(1))

# Variabilité inter-runs : même alerte, répétitions différentes
rep_var = per_alert.groupby(["model", "alert_id"])["total_tokens"].agg(["mean", "std"])
rep_var["cv"] = rep_var["std"] / rep_var["mean"]
between_cv = per_alert.groupby(["model", "alert_id"])["total_tokens"].mean().groupby("model") \
                      .agg(lambda s: s.std() / s.mean())
print("\n=== Variabilité ===")
for m in between_cv.index:
    if True:
        print(f"{m} : CV intra-alerte (entre répétitions) moyen = {rep_var.loc[m, 'cv'].mean():.1%} | "
              f"CV inter-alertes = {between_cv[m]:.1%}")

## 10 Qualité du triage IA par rapport à la vérité terrain
Quatre mesures sont produites :
- verdict : exactitude, précision, rappel, F1, avec la classe positive = malicieux ;
- mapping MITRE : précision, rappel et Jaccard au niveau technique (les sous-techniques sont ramenées à la technique mère) ;
- stabilité : part des alertes dont le verdict est identique sur toutes les répétitions ;
- temps de traitement de l'IA, comparé au temps humain supposé.

La qualité rédactionnelle de l'investigation et du rapport n'est pas mesurable automatiquement. Recommandation : faire noter un échantillon de rapports par 2 analystes en aveugle (grille de 1 à 5 : exactitude, complétude, actionnabilité), puis rapporter l'accord inter-juges (κ de Cohen). Un modèle de grille est exporté dans `human_rating_template.csv`.

In [ ]:
def parse_json(s):
    try:
        return json.loads(s)
    except Exception:
        m = re.search(r"\{.*\}", s or "", re.S)
        try:
            return json.loads(m.group(0)) if m else {}
        except Exception:
            return {}


TECH_RE = re.compile(r"T\d{4}(?:\.\d{3})?")
def techniques(s):
    return {t.split(".")[0] for t in TECH_RE.findall(str(s) if isinstance(s, str) else "")}

LABEL_MAP = {"malicious": 1, "true_positive": 1, "tp": 1, "1": 1, "true": 1, "malveillant": 1,
             "benign": 0, "false_positive": 0, "fp": 0, "0": 0, "false": 0, "benin": 0, "bénin": 0}
def norm_label(x):
    return LABEL_MAP.get(str(x).strip().lower())


outputs["pred"] = outputs["triage"].map(lambda s: norm_label(parse_json(s).get("verdict", "")))
outputs["pred_conf"] = outputs["triage"].map(lambda s: parse_json(s).get("confidence"))
outputs["pred_mitre"] = outputs["mitre"].map(techniques)
print(f"Verdicts non interprétables : {outputs['pred'].isna().mean():.1%}")


def cls_metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    tp = int(((y == 1) & (p == 1)).sum()); tn = int(((y == 0) & (p == 0)).sum())
    fp = int(((y == 0) & (p == 1)).sum()); fn = int(((y == 1) & (p == 0)).sum())
    prec = tp / (tp + fp) if tp + fp else np.nan
    rec = tp / (tp + fn) if tp + fn else np.nan
    f1 = 2 * prec * rec / (prec + rec) if prec and rec else np.nan
    return {"n": len(y), "accuracy": (tp + tn) / len(y), "precision": prec, "recall": rec, "f1": f1,
            "TP": tp, "FP": fp, "TN": tn, "FN": fn}


quality = None
if HAS_LABEL:
    q = outputs.merge(alerts_df[["alert_id", "label"] + (["mitre_gt"] if HAS_MITRE else [])], on="alert_id")
    q["y"] = q["label"].map(norm_label)
    q = q.dropna(subset=["y", "pred"])
    rows = []
    for (m, r), g in q.groupby(["model", "repeat"]):
        row = {"model": m, "repeat": r, **cls_metrics(g["y"].astype(int), g["pred"].astype(int))}
        if HAS_MITRE:
            gt = g["mitre_gt"].map(techniques); pr = g["pred_mitre"]
            inter = sum(len(a & b) for a, b in zip(gt, pr))
            row["mitre_precision"] = inter / max(1, sum(map(len, pr)))
            row["mitre_recall"] = inter / max(1, sum(map(len, gt)))
            row["mitre_jaccard"] = np.mean([len(a & b) / len(a | b) for a, b in zip(gt, pr) if a | b] or [np.nan])
        rows.append(row)
    quality = pd.DataFrame(rows)
    print("=== Qualité par répétition (MESURÉ) ===")
    display(quality.round(3))
    print("=== Moyenne ± écart-type sur les répétitions ===")
    display(quality.drop(columns="repeat").groupby("model").agg(["mean", "std"]).round(3))
else:
    print("⚠️  Pas de colonne 'label' : ajoutez la vérité terrain pour répondre au reviewer 2.")

stab = outputs.groupby(["model", "alert_id"])["pred"].nunique(dropna=False).eq(1).groupby("model").mean()
print("\nStabilité du verdict entre répétitions :", stab.map("{:.1%}".format).to_dict())
lat = per_alert.groupby("model")["latency_s"].mean()
print("Temps de traitement IA moyen par alerte (s) :", lat.round(1).to_dict(),
      f"| hypothèse humaine L1 = {CENTRAL['HUMAN_MIN_TRIAGE_L1']} min, complète = {CENTRAL['HUMAN_MIN_FULL_INVEST']} min")

# Grille d'évaluation humaine en aveugle (à remplir par 2 analystes)
sample = outputs.sample(min(20, len(outputs)), random_state=SEED)[["model", "repeat", "alert_id", "report"]]
sample.assign(rater_1_accuracy="", rater_1_completeness="", rater_1_actionability="",
              rater_2_accuracy="", rater_2_completeness="", rater_2_actionability="") \
      .to_csv(OUT_DIR / "human_rating_template.csv", index=False)

## 10b Justification de la taille d'échantillon
La taille de l'échantillon est justifiée par la précision obtenue, et non par un chiffre rond. Deux grandeurs sont visées, avec des exigences différentes :
- la moyenne de tokens par alerte, base des estimations d'énergie : on calcule le n nécessaire pour une précision de ±5 % (IC95 %) ;
- l'exactitude du triage : on donne l'intervalle de Wilson à 95 %, qui est plus large.

La courbe de convergence montre que la moyenne se stabilise bien avant n = 50.

In [ ]:
from math import sqrt
def wilson(k, n, z=1.96):
    p = k / n; c = (p + z*z/(2*n)) / (1 + z*z/n); h = z*sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
    return c - h, c + h

for m, g in per_alert.groupby("model"):
    t = g.groupby("alert_id")["total_tokens"].mean()          # 1 valeur par alerte (moyenne des répétitions)
    cv = t.std(ddof=1) / t.mean()
    print(f"[{m}] {len(t)} alertes | CV inter-alertes des tokens = {cv:.1%} | "
          f"demi-largeur IC95 % obtenue = ±{1.96*cv/sqrt(len(t)):.1%}")
    for prec in (0.10, 0.05, 0.03):
        print(f"   n requis pour ±{prec:.0%} sur la moyenne de tokens : {int(np.ceil((1.96*cv/prec)**2))}")
    if quality is not None:
        q1 = quality[quality.model == m].iloc[0]
        lo, hi = wilson(q1.TP + q1.TN, q1.n)
        print(f"   exactitude répétition 1 = {(q1.TP+q1.TN)/q1.n:.1%} [Wilson IC95 % : {lo:.1%} – {hi:.1%}] sur n = {q1.n}")
    perm = np.array([RNG.permutation(t.values).cumsum() / np.arange(1, len(t)+1) for _ in range(200)])
    fig, ax = plt.subplots(figsize=(6, 3.2)) if 'plt' in globals() else (None, None)
    if ax is None:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(6, 3.2))
    x = np.arange(1, len(t)+1)
    ax.fill_between(x, np.percentile(perm, 2.5, 0), np.percentile(perm, 97.5, 0), alpha=.3, label="95 % des ordres d'arrivée")
    ax.axhline(t.mean(), color="k", lw=1); ax.axhspan(t.mean()*0.95, t.mean()*1.05, color="g", alpha=.1, label="±5 %")
    ax.set_xlabel("nombre d'alertes"); ax.set_ylabel("tokens moyens / alerte"); ax.set_title(f"Convergence — {m}"); ax.legend()
    fig.tight_layout(); fig.savefig(OUT_DIR / f"fig_sample_size_{m}.png", dpi=300); plt.show()

## 11 Baseline SOAR à règles (comparaison avec l'automatisation conventionnelle)
Un moteur de règles naïf (mots-clés → verdict + techniques) est appliqué aux mêmes alertes. Son temps CPU est mesuré, puis converti en énergie avec "SOAR_CPU_W".

Ce n'est pas un SOAR de production. Cette baseline donne un ordre de grandeur : une automatisation déterministe coûte de l'ordre de quelques microwattheures par alerte, mais elle a une couverture limitée. L'ajout est utile pour situer l'agentique entre l'humain et le SOAR classique.

In [ ]:
SOAR_RULES = {"mimikatz": "T1003", "lsass": "T1003", "powershell": "T1059", "encoded": "T1027",
              "failed login": "T1110", "brute": "T1110", "spraying": "T1110", "phishing": "T1566",
              "macro": "T1204", "exfil": "T1041", "ransom": "T1486", ".lock": "T1486"}

def soar_playbook(text):
    t = text.lower()
    tech = {v for k, v in SOAR_RULES.items() if k in t}
    return (1 if tech else 0), tech

N_LOOP = 2000
t0 = time.process_time()
for _ in range(N_LOOP):
    soar_out = [soar_playbook(a) for a in alerts_df["alert"]]
SOAR_CPU_S_PER_ALERT = (time.process_time() - t0) / (N_LOOP * len(alerts_df))
print(f"Temps CPU SOAR mesuré : {SOAR_CPU_S_PER_ALERT*1e6:.2f} µs/alerte "
      f"→ {SOAR_CPU_S_PER_ALERT*CENTRAL['SOAR_CPU_W']/3600*1e6:.4f} µWh/alerte (ESTIMÉ)")

soar_quality = None
if HAS_LABEL:
    y = alerts_df["label"].map(norm_label)
    mask = y.notna()
    soar_quality = cls_metrics(y[mask].astype(int), np.array([p for p, _ in soar_out])[mask.values])
    if HAS_MITRE:
        gt = alerts_df["mitre_gt"].map(techniques); pr = [t for _, t in soar_out]
        inter = sum(len(a & b) for a, b in zip(gt, pr))
        soar_quality["mitre_recall"] = inter / max(1, sum(map(len, gt)))
    print("Qualité SOAR (même vérité terrain) :", {k: round(v, 3) if isinstance(v, float) else v
                                                    for k, v in soar_quality.items()})

## 12 Énergie IA : deux estimateurs, PUE corrigé, réconciliation explicite 

| Estimateur | Formule | Données mesurées utilisées |
|---|---|---|
| **E1 — par requête** | `n_requêtes × E_QUERY_WH` (PUE déjà inclus dans [6], **non réappliqué**) | nombre de requêtes |
| **E2 — par token** | `(tokens_sortie × e_out + tokens_entrée × e_out × PREFILL_RATIO) / 3600 × PUE` | tokens entrée/sortie |


**Pourquoi E1 et E2 divergent.** Le rapport E2/E1 se décompose exactement en un rapport de tailles de requête :

  "E2/E1 = (tokens effectifs observés par requête) / (tokens effectifs implicites dans le benchmark [6])"

E1 suppose que chaque requête du pipeline a la taille de la requête de référence du benchmark. E2 utilise les tokens réellement mesurés. L'écart n'est donc pas une erreur aléatoire : il mesure à quel point les requêtes du pipeline sont plus longues (ou plus courtes) que celles du benchmark.

**Modèle théorique à 12 requêtes.** Il est calculé à part et étiqueté THÉORIQUE. Il ne doit plus être comparé à E2 du pipeline mesuré pour annoncer un « accord à 4 % ».

In [ ]:
def pue_for(name, p):
    return 1.0 if PARAMS[name].get("includes_pue") else p["PUE"]

def e1_wh(n_queries, p, factor=1.0):
    """factor = rapport du benchmark du modèle à celui de REF_MODEL (Jegham T4, prompt court)."""
    return n_queries * p["E_QUERY_WH"] * factor * pue_for("E_QUERY_WH", p)

def e2_wh(prompt_tokens, completion_tokens, p):
    joules = completion_tokens * p["E_OUT_J_PER_TOKEN"] + prompt_tokens * p["E_OUT_J_PER_TOKEN"] * p["PREFILL_RATIO"]
    return joules / 3600 * pue_for("E_OUT_J_PER_TOKEN", p)

# E1-ajusté : même source que E1 (Jegham et al., Table 4), mis à l'échelle de la taille RÉELLE des requêtes.
# Ajustement exact E = a·tokens_entrée + b·tokens_sortie + c sur les 3 points publiés de CHAQUE modèle (PUE inclus) — voir 3.
pa = per_alert
pa["E1_Wh"] = e1_wh(pa["n_queries"], CENTRAL, pa["e1_factor"])
_fit = np.vstack(pa["api_model"].map(lambda m: JFIT.get(m, JFIT[REF_MODEL])))
pa["E1_scaled_Wh"] = _fit[:, 0] * pa["prompt_tokens"] + _fit[:, 1] * pa["completion_tokens"] + _fit[:, 2] * pa["n_queries"]
pa["E2_Wh"] = e2_wh(pa["prompt_tokens"], pa["completion_tokens"], CENTRAL)
pa["E2_over_E1"] = pa["E2_Wh"] / pa["E1_Wh"]
# Anciennes méthodes (v1) — pour la lettre de réponse uniquement
pa["v1_E1_double_PUE_Wh"] = pa["n_queries"] * CENTRAL["E_QUERY_WH"] * pa["e1_factor"] * 1.2
pa["v1_E2_all_tokens_as_output_Wh"] = pa["total_tokens"] * CENTRAL["E_OUT_J_PER_TOKEN"] / 3600 * CENTRAL["PUE"]

# ---- Décomposition analytique de l'écart ----
pue2 = pue_for("E_OUT_J_PER_TOKEN", CENTRAL)
recon_rows = []
for m, g in pa.groupby("model"):
    implied_tok_per_query = (CENTRAL["E_QUERY_WH"] * g["e1_factor"].iloc[0] * pue_for("E_QUERY_WH", CENTRAL) * 3600
                             / (CENTRAL["E_OUT_J_PER_TOKEN"] * pue2))
    obs_eff = ((g["completion_tokens"] + CENTRAL["PREFILL_RATIO"] * g["prompt_tokens"]) / g["n_queries"]).mean()
    recon_rows.append({
        "model": m, "nature": "ESTIMÉ (à partir de tokens MESURÉS)",
        "E1_Wh/alerte": g["E1_Wh"].mean(), "E1-ajusté (Jegham T4) Wh/alerte": g["E1_scaled_Wh"].mean(),
        "E2_Wh/alerte": g["E2_Wh"].mean(),
        "rapport E2/E1": g["E2_over_E1"].mean(),
        "tokens effectifs observés / requête": obs_eff,
        "tokens effectifs implicites dans [6] / requête": implied_tok_per_query,
        "rapport des tailles (= E2/E1)": obs_eff / implied_tok_per_query,
        "v1 : E1 × 1,2 (PUE compté 2×)": g["v1_E1_double_PUE_Wh"].mean(),
        "v1 : E2 tous tokens au tarif sortie": g["v1_E2_all_tokens_as_output_Wh"].mean()})
reconciliation = pd.DataFrame(recon_rows).set_index("model")
theoretical = pd.DataFrame([{
    "nature": "THÉORIQUE (non mesuré)", "n_requêtes": N_QUERIES_THEORETICAL,
    "E_Wh/alerte": N_QUERIES_THEORETICAL * CENTRAL["E_QUERY_WH"] * pue_for("E_QUERY_WH", CENTRAL)}],
    index=[f"Modèle théorique {N_QUERIES_THEORETICAL} requêtes"])
display(reconciliation.T.round(4))
display(theoretical.round(3))

# ---- Indépendance des estimateurs (R2) ----
s1, s2 = PARAMS["E_QUERY_WH"]["source"].strip(), PARAMS["E_OUT_J_PER_TOKEN"]["source"].strip()
if not s1 or not s2:
    verdict = "INDÉTERMINÉE : renseignez les deux sources."
elif s1 == s2:
    verdict = "NON indépendants (même source) → présenter E2 comme une variante de E1, pas comme une validation croisée."
else:
    verdict = "Indépendants quant aux coefficients (sources distinctes). Ils partagent les mêmes données mesurées (tokens/requêtes)."
print("Indépendance E1/E2 :", verdict)
print("→ Formulation recommandée : présenter [E1 ; E2] comme une FOURCHETTE d'estimation, et non comme un accord.")

## 12b Comparaison des configurations : modèle × architecture (R1.5)
Ce tableau répond à « one model, one pipeline architecture ». Pour chaque configuration, il compare le coût (requêtes, tokens et énergie, MESURÉ puis ESTIMÉ) et la qualité (MESURÉ).

La question à discuter dans le manuscrit : "l'architecture multi-agent apporte-t-elle un gain de qualité qui justifie son surcoût énergétique" par rapport à un agent unique, et un petit modèle suffit-il ?

In [ ]:
comp = per_alert.groupby("model").agg(
    n_requêtes=("n_queries", "mean"), tokens_entrée=("prompt_tokens", "mean"),
    tokens_sortie=("completion_tokens", "mean"), latence_s=("latency_s", "mean"),
    E1_Wh=("E1_Wh", "mean"), E1_ajusté_Wh=("E1_scaled_Wh", "mean"), E2_Wh=("E2_Wh", "mean"))
if quality is not None:
    qcols = [c for c in ["accuracy", "precision", "recall", "f1", "mitre_recall", "mitre_jaccard"] if c in quality]
    comp = comp.join(quality.groupby("model")[qcols].mean())
comp = comp.join(stab.rename("stabilité_verdict"))
comp.index.name = "configuration (modèle_architecture)"
display(comp.round(3))

# Surcoût du multi-agent par rapport à l'agent unique, à modèle égal
for api_model in MODELS:
    a, b = f"{api_model}_multi_agent", f"{api_model}_single_agent"
    if a in comp.index and b in comp.index:
        msg = (f"{api_model} : multi-agent / agent unique → énergie E2 ×{comp.loc[a,'E2_Wh']/comp.loc[b,'E2_Wh']:.2f}, "
               f"E1-ajusté ×{comp.loc[a,'E1_ajusté_Wh']/comp.loc[b,'E1_ajusté_Wh']:.2f}")
        if "f1" in comp:
            msg += f" | ΔF1 = {comp.loc[a,'f1']-comp.loc[b,'f1']:+.3f}"
        print(msg)
comp.to_csv(OUT_DIR / "configuration_comparison.csv")

## 13 Baseline humaine (R1.1, R2)
Deux scénarios humains sont évalués, au lieu d'une valeur unique de 250 min :
- **triage L1** (10–45 min) : comparaison la plus conservatrice ;
- **investigation complète** (120–240 min, médiane 180 min).

`WS_ATTRIBUTION` < 1 représente la charge partagée : un analyste qui mène plusieurs investigations en parallèle n'attribue qu'une fraction de son poste à chaque alerte.


In [ ]:
def human_wh(minutes, p):
    return minutes / 60 * p["WS_POWER_W"] * p["WS_ATTRIBUTION"]

HUMAN_MEASURED = None
if Path(HUMAN_MEASUREMENTS_PATH).exists():
    hm = pd.read_csv(HUMAN_MEASUREMENTS_PATH)
    hm["wh"] = hm["measured_wh"] if "measured_wh" in hm.columns else human_wh(hm["minutes"], CENTRAL)
    HUMAN_MEASURED = hm
    print("Mesures humaines chargées (MESURÉ) :")
    display(hm[["minutes", "wh"]].apply(describe).T.round(2))
else:
    print("Pas de mesures humaines → baseline = HYPOTHÈSE (plages du §3). Le signaler explicitement dans le manuscrit.")

for name in ["HUMAN_MIN_TRIAGE_L1", "HUMAN_MIN_FULL_INVEST", "HUMAN_REVIEW_MIN"]:
    lo, c, hi = (human_wh(PARAMS[name][k], CENTRAL) for k in ("low", "central", "high"))
    print(f"{name:24s}: {c:7.1f} Wh/alerte (plage {lo:.1f}–{hi:.1f}, à puissance et attribution centrales)")
print(f"Rappel v1 : 250 min × 120 W = {250/60*120:.0f} Wh (valeur non justifiée, abandonnée)")

## 14 Scénarios comparés : énergie et CO₂ opérationnels par alerte (R1.4, R2)

Périmètre : **émissions opérationnelles liées à l'électricité uniquement**. L'empreinte de cycle de vie (fabrication des GPU et des postes, réseau, eau) est **exclue** et doit être discutée comme une limite.

Le scénario **hybride** (IA + relecture humaine résiduelle) est le scénario réaliste 
Le scénario « IA seule » est une borne optimiste.

In [ ]:
def scenarios(p, tok):
    """p : dict de paramètres (scalaires ou tableaux) ; tok : dict de tableaux mesurés par alerte. Renvoie (Wh, gCO2e)."""
    ci_c, ci_w = p["CI_CLOUD"] / 1000, p["CI_WORKSTATION"] / 1000
    ai1 = e1_wh(tok["n_queries"], p, tok.get("e1_factor", 1.0))
    ai2 = e2_wh(tok["prompt_tokens"], tok["completion_tokens"], p)
    cli = tok["latency_s"] / 3600 * p["CLIENT_W"]
    rev = human_wh(p["HUMAN_REVIEW_MIN"], p)
    soar = SOAR_CPU_S_PER_ALERT / 3600 * p["SOAR_CPU_W"]
    hl1, hfull = human_wh(p["HUMAN_MIN_TRIAGE_L1"], p), human_wh(p["HUMAN_MIN_FULL_INVEST"], p)
    energy = {  # (cloud_Wh, local_Wh)
        "Humain – triage L1":                  (0, hl1),
        "Humain – investigation complète":     (0, hfull),
        "IA seule (E1)":                       (ai1, cli),
        "IA seule (E2)":                       (ai2, cli),
        "Hybride IA (E1) + relecture humaine": (ai1, cli + rev),
        "Hybride IA (E2) + relecture humaine": (ai2, cli + rev),
        "SOAR à règles (baseline naïf)":       (0, soar),
    }
    if HUMAN_MEASURED is not None:
        energy["Humain (mesuré)"] = (0, tok["human_measured_wh"])
    wh = {k: c + l for k, (c, l) in energy.items()}
    co2 = {k: c * ci_c + l * ci_w for k, (c, l) in energy.items()}
    return wh, co2


def tok_arrays(df, idx=None):
    d = df if idx is None else df.iloc[idx]
    t = {c: d[c].to_numpy(float) for c in ["n_queries", "prompt_tokens", "completion_tokens", "latency_s", "e1_factor"]}
    if HUMAN_MEASURED is not None:
        hw = HUMAN_MEASURED["wh"].to_numpy(float)
        t["human_measured_wh"] = hw.mean() if idx is None else RNG.choice(hw, len(d))
    return t


NATURE = {"Humain (mesuré)": "MESURÉ", "SOAR à règles (baseline naïf)": "ESTIMÉ (CPU mesuré)"}
central_rows = []
for m, g in per_alert.groupby("model"):
    wh, co2 = scenarios(CENTRAL, tok_arrays(g))
    for k in wh:
        nat = NATURE.get(k, "HYPOTHÈSE" if k.startswith("Humain") else "ESTIMÉ (tokens MESURÉS)")
        central_rows.append({"model": m, "scénario": k, "nature": nat,
                             "Wh/alerte": np.mean(wh[k]), "gCO2e/alerte": np.mean(co2[k])})
central_table = pd.DataFrame(central_rows)
display(central_table.style.format({"Wh/alerte": "{:.4g}", "gCO2e/alerte": "{:.4g}"}))

## 15 Propagation de l'incertitude : Monte Carlo et bootstrap (R1.4)
Chaque tirage combine deux sources d'aléa :
- tous les paramètres du 3 sont tirés selon une loi triangulaire (`low`, `central`, `high`) ;
- une alerte mesurée est tirée au hasard (bootstrap), ce qui couvre la variabilité entre alertes et entre répétitions.

Le résultat est une médiane avec un intervalle à 95 % pour chaque scénario et pour chaque réduction relative. `P(réduction > 0)` indique la probabilité que l'IA émette moins que l'humain sous les hypothèses retenues.

In [ ]:
N_MC = 20_000

def sample_params(n):
    out = {}
    for k, v in PARAMS.items():
        lo, c, hi = v["low"], v["central"], v["high"]
        out[k] = np.full(n, c, float) if lo == hi else RNG.triangular(lo, c, hi, n)
    return out

def pct(x):
    return pd.Series({"p2.5": np.percentile(x, 2.5), "médiane": np.median(x), "p97.5": np.percentile(x, 97.5)})

COMPARISONS = [  # (IA, humain)
    ("Hybride IA (E2) + relecture humaine", "Humain – triage L1"),
    ("Hybride IA (E1) + relecture humaine", "Humain – triage L1"),
    ("Hybride IA (E2) + relecture humaine", "Humain – investigation complète"),
    ("IA seule (E2)", "Humain – investigation complète"),
    ("IA seule (E1)", "Humain – investigation complète"),
]
if HUMAN_MEASURED is not None:
    COMPARISONS.insert(0, ("Hybride IA (E2) + relecture humaine", "Humain (mesuré)"))

mc_co2, mc_red, mc_annual, MC_DRAWS = [], [], [], {}
for m, g in per_alert.groupby("model"):
    p = sample_params(N_MC)
    idx = RNG.integers(0, len(g), N_MC)
    wh, co2 = scenarios(p, tok_arrays(g, idx))
    MC_DRAWS[m] = co2
    for k, v in co2.items():
        mc_co2.append({"model": m, "scénario": k, **pct(v)})
    for ai, hu in COMPARISONS:
        red = 100 * (1 - co2[ai] / co2[hu])
        mc_red.append({"model": m, "IA": ai, "référence": hu, **pct(red), "P(réduction>0)": (red > 0).mean()})
        saved_kg = p["ALERTS_PER_YEAR"] * (co2[hu] - co2[ai]) / 1000
        mc_annual.append({"model": m, "IA": ai, "référence": hu, "nature": "EXTRAPOLÉ", **pct(saved_kg)})

mc_co2 = pd.DataFrame(mc_co2)
mc_red = pd.DataFrame(mc_red)
mc_annual = pd.DataFrame(mc_annual)
print("=== gCO2e par alerte (opérationnel, électricité) : médiane [IC95 %] ===")
display(mc_co2.round(5))
print("=== Réduction relative (%) : médiane [IC95 %] ===")
display(mc_red.round(2))

## 16 Analyse de sensibilité un-à-un : tornado (R1.4)
Indicateur étudié : la réduction de CO₂ du scénario hybride (E2) par rapport au triage humain L1, soit la comparaison la plus défavorable à l'IA. Chaque paramètre est placé à sa borne basse puis haute, les autres restant à leur valeur centrale.

In [ ]:
import matplotlib.pyplot as plt

AI_REF, HU_REF = "Hybride IA (E2) + relecture humaine", "Humain – triage L1"
REF_CONFIG = f"{MODELS[0]}_{ARCHITECTURES[0]}"
_ref = per_alert[per_alert["model"] == REF_CONFIG]
tok_c = {c: _ref[c].mean() for c in ["n_queries", "prompt_tokens", "completion_tokens", "latency_s", "e1_factor"]}
print("Configuration de référence pour le tornado :", REF_CONFIG)
if HUMAN_MEASURED is not None:
    tok_c["human_measured_wh"] = HUMAN_MEASURED["wh"].mean()

def reduction(p):
    _, co2 = scenarios(p, tok_c)
    return 100 * (1 - co2[AI_REF] / co2[HU_REF])

base = reduction(CENTRAL)
tornado = []
for k, v in PARAMS.items():
    if v["low"] == v["high"]:
        continue
    lo, hi = reduction({**CENTRAL, k: v["low"]}), reduction({**CENTRAL, k: v["high"]})
    if abs(hi - lo) > 1e-9:
        tornado.append({"paramètre": k, "réduction @low": lo, "réduction @high": hi, "amplitude": abs(hi - lo)})
tornado = pd.DataFrame(tornado).sort_values("amplitude")
print(f"Réduction centrale ({AI_REF} vs {HU_REF}) : {base:.2f} %")
display(tornado.sort_values("amplitude", ascending=False).round(3))

fig, ax = plt.subplots(figsize=(7, 0.45 * len(tornado) + 1.2))
for i, r in enumerate(tornado.itertuples()):
    a, b = sorted([r[2], r[3]])
    ax.barh(i, b - a, left=a, color="#4C72B0")
ax.axvline(base, color="k", lw=1)
ax.set_yticks(range(len(tornado)), tornado["paramètre"])
ax.set_xlabel("Réduction de CO₂ (%) — hybride (E2) vs triage humain L1")
ax.set_title("Sensibilité un-à-un (bornes low/high du §3)")
fig.tight_layout(); fig.savefig(OUT_DIR / "fig_tornado.png", dpi=300); plt.show()

## 17 Scénarios de réseau électrique : cloud et poste séparés (R1.5, R2)


In [ ]:
GRID_LEVELS = {"bas carbone": 50, "intermédiaire": 250, "Azure (350)": 350, "moyenne mondiale IEA 2025 (435)": 435,
               "Maroc (536)": 536, "fortement carboné": 750}
grid_rows = []
for m, g in per_alert.groupby("model"):
    tok = tok_arrays(g)
    for cn, cc in GRID_LEVELS.items():
        for wn, wc in GRID_LEVELS.items():
            _, co2 = scenarios({**CENTRAL, "CI_CLOUD": cc, "CI_WORKSTATION": wc}, tok)
            grid_rows.append({"model": m, "réseau cloud": cn, "réseau SOC": wn,
                              "hybride E2 (g)": np.mean(co2[AI_REF]), "humain L1 (g)": np.mean(co2[HU_REF]),
                              "réduction %": 100 * (1 - np.mean(co2[AI_REF]) / np.mean(co2[HU_REF]))})
grid = pd.DataFrame(grid_rows)
display(grid.pivot_table(index=["model", "réseau cloud"], columns="réseau SOC", values="réduction %").round(1))

## 18 Économies annuelles (EXTRAPOLÉ, R1.4)
Extrapolation : `ALERTS_PER_YEAR × (CO₂ humain − CO₂ IA)`. Le volume annuel est lui-même incertain (plage du 3).

À présenter en kgCO₂e/an avec intervalle.

In [ ]:
display(mc_annual.round(1))

## 19 Extended reasoning (PROJECTION, R2)


In [ ]:
proj = []
for m, g in per_alert.groupby("model"):
    for lvl in ["low", "central", "high"]:
        mult = PARAMS["REASONING_TOKEN_MULT"][lvl]
        e = e2_wh(g["prompt_tokens"], g["completion_tokens"] * mult, CENTRAL).mean()
        proj.append({"model": m, "nature": "PROJECTION", "multiplicateur": mult,
                     "Wh/alerte (E2)": e, "gCO2e/alerte": e * CENTRAL["CI_CLOUD"] / 1000})
projection = pd.DataFrame(proj)
display(projection.round(4))

## 20 Figures pour le manuscrit

In [ ]:
for m, g in per_alert.groupby("model"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    a = agent_stats.loc[m]
    axes[0].bar(a.index, a["prompt_tokens"], label="entrée (prompt)", color="#8DA0CB")
    axes[0].bar(a.index, a["completion_tokens"], bottom=a["prompt_tokens"], label="sortie (completion)", color="#FC8D62")
    axes[0].set_ylabel("tokens moyens / appel"); axes[0].set_title(f"Tokens mesurés par agent — {m}"); axes[0].legend()
    d = MC_DRAWS[m]
    names = list(d.keys())
    axes[1].boxplot([d[k] for k in names], vert=False, showfliers=False)
    axes[1].set_yticks(range(1, len(names) + 1), names); axes[1].set_xscale("log")
    axes[1].set_xlabel("gCO₂e par alerte (log) — Monte Carlo, IC95 %")
    axes[1].set_title("Émissions opérationnelles par scénario")
    fig.tight_layout(); fig.savefig(OUT_DIR / f"fig_tokens_co2_{m}.png", dpi=300); plt.show()

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
FIG_DIR = OUT_DIR / "figures_paper"; FIG_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif", "Times New Roman", "DejaVu Serif"],
                     "mathtext.fontset": "stix", "font.size": 8, "axes.spines.top": False, "axes.spines.right": False,
                     "pdf.fonttype": 42, "axes.linewidth": 0.6})
BLUE, LIGHT, ORANGE, GREY = "#2f6db0", "#9bbbe0", "#d08c2f", "#6b6b6b"
def label_of(cfg):
    m, a = _split_config(cfg)
    return ("GPT-4o mini" if "mini" in m else "GPT-4o") + (", four agents" if a == "multi_agent" else ", single agent")
def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight"); fig.savefig(FIG_DIR / f"{name}.png", dpi=300, bbox_inches="tight"); plt.show()

# Fig. 1 — precision of the mean token count vs. number of alerts (from the measured between-alert CV of each configuration)
fig, ax = plt.subplots(figsize=(3.45, 2.3))
n = np.arange(3, 101)
for cfg, g in per_alert.groupby("model"):
    t = g.groupby("alert_id")["total_tokens"].mean()
    cv = 100 * t.std(ddof=1) / t.mean()
    _, arch = _split_config(cfg)
    ax.plot(n, 1.96 * cv / np.sqrt(n), color=ORANGE if "mini" in cfg else BLUE, ls="-" if arch == "multi_agent" else "--",
            lw=1.2, label=f"{label_of(cfg)} (CV {cv:.1f}%)")
ax.axhline(5, color="#555555", lw=0.7); ax.text(99, 5.25, "±5% target", ha="right", va="bottom", fontsize=6.5, color="#555555")
ax.axvline(len(t), color="#999999", lw=0.6, ls=":"); ax.text(len(t) + 1, 15.2, f"{len(t)} alerts", fontsize=6.5, color="#555555")
ax.set_xlim(3, 100); ax.set_ylim(0, 16)
ax.set_xlabel("Number of alerts"); ax.set_ylabel("95% CI half-width of mean tokens (%)")
ax.legend(fontsize=6, frameon=False, loc="upper right", bbox_to_anchor=(1.0, 0.93)); ax.grid(color="#e3e3e3", lw=0.5); ax.set_axisbelow(True)
fig.tight_layout(pad=0.3); save(fig, "fig_sample_size")

# Fig. 2 — measured tokens per agent, four-agent pipelines
multi = [c for c in agent_stats.index.get_level_values("model").unique() if c.endswith("_multi_agent")]
fig, axes = plt.subplots(1, len(multi), figsize=(3.45, 2.2), sharey=True, squeeze=False)
for ax, cfg in zip(axes[0], multi):
    a = agent_stats.loc[cfg].reindex(AGENTS); x = np.arange(len(a))
    ax.bar(x, a["prompt_tokens"], color=LIGHT, width=0.65, label="input tokens")
    ax.bar(x, a["completion_tokens"], bottom=a["prompt_tokens"], color=BLUE, width=0.65, label="output tokens")
    for k, (pi, co, sh) in enumerate(zip(a["prompt_tokens"], a["completion_tokens"], a["share_of_total_%"])):
        ax.text(k, pi + co + 18, f"{sh:.1f}%", ha="center", fontsize=6.5)
    ax.set_xticks(x, ["Triage", "MITRE", "Investig.", "Report."], fontsize=6.3)
    ax.set_title("GPT-4o mini" if "mini" in cfg else "GPT-4o", fontsize=8); ax.grid(axis="y", color="#e3e3e3", lw=0.5); ax.set_axisbelow(True)
axes[0][0].set_ylabel("Tokens per alert (mean)"); axes[0][0].legend(fontsize=6.3, frameon=False, loc="upper left")
fig.tight_layout(pad=0.3); save(fig, "fig_agents")

# Fig. 3 — operational emissions per alert (Monte Carlo median and 95% interval)
ref, single = REF_CONFIG, REF_CONFIG.replace("multi_agent", "single_agent")
rows = [("Human, full\ninvestigation", MC_DRAWS[ref]["Humain – investigation complète"], GREY),
        ("Human, L1 triage", MC_DRAWS[ref]["Humain – triage L1"], GREY),
        ("Hybrid: 4-agent AI (E2)\n+ human review", MC_DRAWS[ref]["Hybride IA (E2) + relecture humaine"], ORANGE),
        ("4-agent AI only (E1)", MC_DRAWS[ref]["IA seule (E1)"], BLUE),
        ("4-agent AI only (E2)", MC_DRAWS[ref]["IA seule (E2)"], BLUE)]
if single in MC_DRAWS:
    rows.append(("1-agent AI only (E2)", MC_DRAWS[single]["IA seule (E2)"], BLUE))
fig, ax = plt.subplots(figsize=(3.45, 2.4))
for i, (lab, d, c) in enumerate(rows[::-1]):
    lo, med, hi = np.percentile(d, [2.5, 50, 97.5])
    ax.plot([lo, hi], [i, i], color=c, lw=1.4, solid_capstyle="butt"); ax.plot([med], [i], "o", color=c, ms=3.6, zorder=3)
    ax.text(hi * 1.25, i, f"{med:.3g}", va="center", fontsize=7, color="#333333")
ax.set_yticks(range(len(rows)), [r[0] for r in rows[::-1]]); ax.set_xscale("log"); ax.set_xlim(0.05, 900)
ax.set_xticks([0.1, 1, 10, 100], ["0.1", "1", "10", "100"]); ax.set_xlabel("gCO$_2$e per alert (log scale)")
ax.grid(axis="x", color="#e3e3e3", lw=0.5); ax.set_axisbelow(True)
fig.tight_layout(pad=0.3); save(fig, "fig_co2_scenarios")

# Fig. 4 — one-at-a-time sensitivity (hybrid E2 vs L1 triage, reference configuration)
NAMES = {"HUMAN_REVIEW_MIN": ("Human review time", "min"), "HUMAN_MIN_TRIAGE_L1": ("L1 triage time", "min"),
         "CI_WORKSTATION": ("Workstation grid", "g/kWh"), "WS_POWER_W": ("Workstation power", "W"),
         "WS_ATTRIBUTION": ("Workload attribution", ""), "E_OUT_J_PER_TOKEN": ("Energy per output token", "J"),
         "CI_CLOUD": ("Cloud grid", "g/kWh"), "PUE": ("PUE", ""), "CLIENT_W": ("Client power", "W"), "PREFILL_RATIO": ("Input/output ratio", "")}
def tlabel(p):
    name, u = NAMES.get(p, (p, ""))
    return f"{name}\n({PARAMS[p]['low']:g}–{PARAMS[p]['high']:g}{' ' + u if u else ''})"
top = tornado.sort_values("amplitude", ascending=False).head(8).iloc[::-1]
fig, ax = plt.subplots(figsize=(3.45, 3.0))
for i, r in enumerate(top.itertuples(index=False)):
    p_, lo_, hi_ = r[0], r[1], r[2]
    ax.barh(i, lo_ - base, left=base, color=LIGHT, height=0.6); ax.barh(i, hi_ - base, left=base, color=BLUE, height=0.6)
ax.axvline(base, color="#222222", lw=0.7); ax.axvline(0, color="#999999", lw=0.5, ls="--")
ax.set_yticks(range(len(top)), [tlabel(p) for p in top["paramètre"]], fontsize=6.5)
ax.set_xlabel("CO$_2$e reduction vs. L1 triage (%)")
ax.legend(handles=[Patch(color=LIGHT, label="parameter at low bound"), Patch(color=BLUE, label="parameter at high bound")],
          fontsize=6.5, frameon=False, loc="upper center", bbox_to_anchor=(0.4, -0.22), ncol=2)
fig.tight_layout(pad=0.3); save(fig, "fig_tornado")
print("Figures de l'article enregistrées dans", FIG_DIR.resolve())
for f in sorted(FIG_DIR.iterdir()): print("  ", f.name)

In [ ]:
per_alert.to_csv(OUT_DIR / "per_alert.csv", index=False)
token_stats.to_csv(OUT_DIR / "token_stats.csv")
agent_stats.to_csv(OUT_DIR / "agent_stats.csv")
reconciliation.to_csv(OUT_DIR / "reconciliation.csv")
theoretical.to_csv(OUT_DIR / "theoretical_model.csv")
central_table.to_csv(OUT_DIR / "central_estimates.csv", index=False)
mc_co2.to_csv(OUT_DIR / "mc_co2.csv", index=False)
mc_red.to_csv(OUT_DIR / "mc_reductions.csv", index=False)
mc_annual.to_csv(OUT_DIR / "mc_annual.csv", index=False)
tornado.to_csv(OUT_DIR / "tornado.csv", index=False)
grid.to_csv(OUT_DIR / "grid_scenarios.csv", index=False)
projection.to_csv(OUT_DIR / "projection_reasoning.csv", index=False)
if quality is not None:
    quality.to_csv(OUT_DIR / "quality.csv", index=False)
if soar_quality is not None:
    (OUT_DIR / "soar_quality.json").write_text(json.dumps(soar_quality, indent=2, default=float), encoding="utf-8")
(OUT_DIR / "params.json").write_text(json.dumps(
    {"params": PARAMS, "n_queries_theoretical": N_QUERIES_THEORETICAL, "n_monte_carlo": N_MC}, indent=2, ensure_ascii=False), encoding="utf-8")


def _ver(pkg):
    try:
        from importlib.metadata import version
        return version(pkg)
    except Exception:
        return None


env = {"run_id": RUN_ID, "dry_run": DRY_RUN, "date_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
       "python": sys.version, "platform": platform.platform(),
       "packages": {p: _ver(p) for p in ["openai", "pandas", "numpy", "matplotlib"]},
       "models_requested": MODELS, "architectures": ARCHITECTURES, "energy_benchmark_per_model": JEGHAM_T4, "models_returned": sorted(calls["model_returned"].unique().tolist()),
       "system_fingerprints": sorted(calls["system_fingerprint"].dropna().unique().tolist()),
       "n_alerts": len(alerts_df), "n_repeats": N_REPEATS, "dataset_sha256": dataset_sha256,
       "n_llm_calls": len(calls), "n_failures": sum(1 for _ in open(FAILURE_LOG, encoding="utf-8")) if FAILURE_LOG.exists() else 0,
       "seed": SEED, "temperature": TEMPERATURE, "top_p": TOP_P, "max_tokens": MAX_TOKENS,
       "prompt_version": PROMPT_VERSION}
(OUT_DIR / "environment.json").write_text(json.dumps(env, indent=2), encoding="utf-8")


def to_latex(df, caption, label, fmt="{:.4g}"):
    df = df.reset_index() if not isinstance(df.index, pd.RangeIndex) else df
    esc = lambda s: str(s).replace("%", r"\%").replace("_", r"\_").replace("&", r"\&").replace("#", r"\#")
    cell = lambda x: esc(fmt.format(x)) if isinstance(x, (float, np.floating)) else esc(x)
    lines = [r"\begin{table}[!t]", r"\centering", rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             r"\begin{tabular}{" + "l" * df.shape[1] + "}", r"\hline",
             " & ".join(esc(c) for c in df.columns) + r" \\", r"\hline"]
    lines += [" & ".join(cell(x) for x in row) + r" \\" for row in df.itertuples(index=False)]
    lines += [r"\hline", r"\end{tabular}", r"\end{table}", ""]
    return "\n".join(lines)


tex = [to_latex(token_stats.reset_index()[lambda d: d.metric.isin(["total_tokens", "prompt_tokens", "completion_tokens"])],
                "Measured token usage per alert (mean, SD, bootstrap 95\\% CI)", "tab:tokens"),
       to_latex(reconciliation.T.reset_index(), "Reconciliation of query- and token-based energy estimators", "tab:recon"),
       to_latex(mc_red.drop(columns="model"), "CO$_2$ reduction (\\%) with Monte Carlo 95\\% interval", "tab:reduction"),
       to_latex(mc_annual.drop(columns="model"), "Extrapolated annual savings (kgCO$_2$e/yr)", "tab:annual")]
if quality is not None:
    tex.insert(1, to_latex(quality.drop(columns="model"), "Triage quality against ground truth, per repeat", "tab:quality", "{:.3f}"))
(OUT_DIR / "tables.tex").write_text("\n".join(tex), encoding="utf-8")

print("Fichiers exportés dans", OUT_DIR.resolve())
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:32s} {f.stat().st_size/1024:8.1f} Ko")